In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('raw',), ('ref',)]

In [2]:
from pyiceberg.transforms import IdentityTransform
transactions = catalog.load_table("raw.transactions")
with transactions.update_sort_order() as order:
    order.asc("from_address", IdentityTransform())
transactions.sort_order()

SortOrder(SortField(source_id=5, transform=IdentityTransform(), direction=SortDirection.ASC, null_order=NullOrder.NULLS_LAST), order_id=1)

In [3]:
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual, LessThan
def day(d):
    return And(GreaterThanOrEqual("block_timestamp", f"2026-10-0{d}T00:00:00"), LessThan("block_timestamp", f"2026-10-0{d + 1}T00:00:00"))

def rewrite(d, sort):
    rows = transactions.scan(row_filter=day(d)).to_arrow()
    if sort:
        rows = rows.sort_by("from_address")
    with transactions.transaction() as tx:
        tx.delete(delete_filter=day(d))
        for i in range(0, rows.num_rows, 250_000):
            tx.append(rows.slice(i, 250_000))

rewrite(1, sort=True)   # 1 Oct: files sorted by from_address
rewrite(2, sort=False)  # 2 Oct: same file size, not sorted

In [4]:
files = transactions.inspect.files().to_pandas()
files["from_min"] = files.lower_bounds.map(lambda b: dict(b)[5].decode()[:10])
files["from_max"] = files.upper_bounds.map(lambda b: dict(b)[5].decode()[:10])
files[["partition", "record_count", "from_min", "from_max"]].head(12)

,partition,record_count,from_min,from_max
0,{'block_day': 2026-10-02},125646,0x00000000,0xfffbe6be
1,{'block_day': 2026-10-02},250000,0x00000000,0xffffbbdc
2,{'block_day': 2026-10-02},250000,0x00000000,0xffffffb5
3,{'block_day': 2026-10-02},250000,0x00000000,0xfffffffd
4,{'block_day': 2026-10-02},250000,0x00000000,0xffffffb5
5,{'block_day': 2026-10-02},250000,0x00000000,0xffff14fe
6,{'block_day': 2026-10-02},250000,0x00000000,0xffffef30
7,{'block_day': 2026-10-02},250000,0x00000000,0xffffb802
8,{'block_day': 2026-10-01},36041,0xfab18fb6,0xffffffb5
9,{'block_day': 2026-10-01},250000,0xda41979b,0xfab18fb6


In [5]:
busy = transactions.scan(row_filter=day(1), selected_fields=("from_address",)).to_duckdb("t").sql(
    "select from_address from t group by 1 order by count(*) desc limit 1").fetchone()[0]
sorted_day = len(transactions.scan(row_filter=And(day(1), EqualTo("from_address", busy))).plan_files())
unsorted_day = len(transactions.scan(row_filter=And(day(2), EqualTo("from_address", busy))).plan_files())
busy, sorted_day, unsorted_day

('0x559432e18b281731c054cd703d4b49872be4ed53', 1, 8)